# Hong Kong Market Support 香港市场支持 (HKEX)

QuantEval now supports the Hong Kong stock market! This notebook demonstrates how to:
1. Load HK stock data (e.g., Tencent 00700).
2. Load HK index data (e.g., Hang Seng Index ^HSI).
3. Apply `HKTransactionCost` which includes bilateral stamp duty, SFC levy, and HKEX trading fees.

QuantEval 现已支持港股市场！本 Notebook 演示：
1. 如何加载港股数据（示例：腾讯 00700）。
2. 如何加载香港指数（示例：恒生指数 ^HSI）。
3. 如何使用 `HKTransactionCost`（包含双边印花税、SFC 征费与港交所手续费）。

In [ ]:
# Import required modules / 导入所需模块
from quanteval import DataLoader, Backtester, DualMAStrategy, HKTransactionCost

## 1. Loading Data / 加载数据

We use `DataLoader.load_hk_stock()` for HKEX stocks (use 5-digit zero-padded symbols) and `DataLoader.load_hk_index()` for indices (via yfinance).

我们使用 `DataLoader.load_hk_stock()` 加载港股（请使用 5 位补零代码），使用 `DataLoader.load_hk_index()` 加载指数（通过 yfinance）。

In [ ]:
# Initialize DataLoader (use cache to avoid repeated downloads) / 初始化 DataLoader（使用缓存以避免重复下载）
loader = DataLoader(use_cache=True)

# Load Tencent (00700) data for 2023 / 加载腾讯（00700）2023 年数据
stock_data = loader.load_hk_stock('00700', start_date='20230101', end_date='20231231')
print("Tencent (00700) Data:")
print(stock_data.head())

# Load Hang Seng Index (^HSI) / 加载恒生指数 (^HSI)
index_data = loader.load_hk_index('^HSI', start_date='20230101', end_date='20231231')
print("\nHang Seng Index (^HSI) Data:")
print(index_data.head())

## 2. Backtesting with HK Transaction Costs / 港股交易成本回测

The Hong Kong market has different transaction costs compared to A-shares. Notably, stamp duty is applied to **both** buy and sell trades (bilateral). `HKTransactionCost` handles this automatically.

港股与 A 股的交易成本不同，尤其是印花税在买卖双方均会征收（双边）。`HKTransactionCost` 会自动为你计算这些费用。

In [ ]:
# Create strategy / 创建策略
strategy = DualMAStrategy(fast_window=10, slow_window=30)

# Set up transaction costs for HK / 设置港股交易成本
hk_costs = HKTransactionCost()

# Run Backtest / 运行回测
bt = Backtester(
    strategy=strategy,
    data=stock_data,
    transaction_costs=hk_costs,
    initial_capital=100000.0,
)
results = bt.run()

## 3. Results and Visualization / 结果与可视化

Let's inspect summary metrics and visualize the equity curve. The example keeps interactive plotting enabled by default — in CI we typically use static output.

查看回测汇总指标并可视化收益曲线。示例默认开启交互式绘图；在 CI 中我们通常使用静态输出以避免图形后端问题。

In [ ]:
# Summary metrics / 汇总指标
print(results.summary())

In [ ]:
# Plot the equity curve (interactive plot) / 可视化收益曲线（交互式）
# For CI or static examples, set interactive=False if needed / 在 CI 或静态示例中可设置 interactive=False
results.plot()